# Nemotron on NVIDIA's API

This notebook calls an NVIDIA **Nemotron** model through NVIDIA's OpenAI-compatible
endpoint (`https://integrate.api.nvidia.com/v1`). Usage is paid for by the **Brev credits
NVIDIA gives your team**, not by Keeboh's compute allowance: the model runs on NVIDIA's
side, while this notebook runs on your team's Keeboh GPU.

**Before you run it**

1. Get your team's NVIDIA API key (from the Brev credits NVIDIA gave you).
2. In your workspace, create a file named `.env` with this line:
   `NVIDIA_API_KEY=nvapi-...` (your key). The next cell reads it.

> Never type the key into a notebook or source file. This workspace is shared with your
> whole team, and notebooks and source files go into your team's submission. A `.env` file
> stays out of it.

The `openai` Python package used below was added to the Keeboh notebook image for this
example (pinned to `openai==3.22.0`); you don't need to install it.

In [ ]:
import os
from pathlib import Path


def read_env(path):
    """The KEY=value lines of a .env file."""
    values = {}
    if path.is_file():
        for line in path.read_text().splitlines():
            name, sep, value = line.strip().partition("=")
            if sep and not name.startswith("#"):
                values[name.strip()] = value.strip().strip('"').strip("'")
    return values


# Your key, from the .env file at the top of your workspace (or the environment).
workspace = Path(os.environ.get("KEEBOH_WORKSPACE_DIR", "/workspace"))
NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY") or read_env(workspace / ".env").get(
    "NVIDIA_API_KEY", ""
)

BASE_URL = "https://integrate.api.nvidia.com/v1"
# A general-purpose Nemotron model. The cell after the next one lists the others.
MODEL = "nvidia/nemotron-3-super-120b-a12b"

if not NVIDIA_API_KEY:
    print("Add NVIDIA_API_KEY=... to the .env file in your workspace, then run this cell again.")

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=NVIDIA_API_KEY)

completion = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are a concise assistant for an agricultural remote-sensing hackathon.",
        },
        {
            "role": "user",
            "content": (
                "In five bullet points, which satellite and field signals could help tell "
                "silage corn from grain corn?"
            ),
        },
    ],
    temperature=0.3,
    max_tokens=600,
    stream=True,
)

for chunk in completion:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="")

## Other Nemotron models

NVIDIA's catalogue changes over time. This lists the Nemotron models the endpoint offers
right now; set `MODEL` above to any of them.

In [ ]:
nemotron = sorted(model.id for model in client.models.list() if "nemotron" in model.id.lower())
print("\n".join(nemotron))